# 03 - Nominal Stress Method Benchmark

Validate the Nominal Stress Method against IIW published examples.

**IIW benchmark case:** Butt weld (FAT 80), steel, Δσ = 80 MPa at N = 2×10⁶ cycles.
Expected result: exactly at FAT class reference point → safety factor = 1.0.

In [ ]:
from weldfatigue.fatigue.nominal_stress import NominalStressAssessment
from weldfatigue.fatigue.sn_curve import SNCurve
from weldfatigue.reporting.plots import FatiguePlots

## Benchmark 1: FAT 80 at Reference Point

In [ ]:
assessor = NominalStressAssessment(fat_class=80, material="steel")
result = assessor.evaluate(stress_range=80.0, num_cycles=2_000_000)

print(f"Method:          {result.method}")
print(f"FAT class:       {result.fat_class}")
print(f"Stress range:    {result.stress_range:.1f} MPa")
print(f"Applied cycles:  {result.applied_cycles:.2e}")
print(f"Allowable cycles:{result.allowable_cycles:.2e}")
print(f"Safety factor:   {result.safety_factor:.3f}")
print(f"Status:          {result.status}")

## Benchmark 2: Higher Stress → Fewer Allowable Cycles

In [ ]:
result_high = assessor.evaluate(stress_range=120.0, num_cycles=2_000_000)
print(f"Δσ = 120 MPa → N_allow = {result_high.allowable_cycles:.2e} → {result_high.status}")

result_low = assessor.evaluate(stress_range=60.0, num_cycles=2_000_000)
print(f"Δσ =  60 MPa → N_allow = {result_low.allowable_cycles:.2e} → {result_low.status}")

## Visualize: S-N Curve with Operating Points

In [ ]:
import matplotlib.pyplot as plt

sn = SNCurve(fat_class=80, material_type="steel")
fig = FatiguePlots.sn_curve_matplotlib(
    sn,
    highlight_point=(2e6, 80),
    title="FAT 80 - Nominal Stress Benchmark"
)
ax = fig.axes[0]
ax.plot(2e6, 120, 'rs', markersize=10, label='Δσ=120 (FAIL)')
ax.plot(2e6, 60, 'g^', markersize=10, label='Δσ=60 (PASS)')
ax.legend()
plt.show()

## Benchmark 3: Variable Amplitude Spectrum

In [ ]:
spectrum = [
    (120.0, 100_000),
    (100.0, 500_000),
    (80.0, 1_000_000),
    (60.0, 2_000_000),
    (40.0, 5_000_000),
]

va_result = assessor.evaluate_spectrum(spectrum)
print(f"Total Miner damage: {va_result.total_damage:.4f}")
print(f"Status: {va_result.status}")
print(f"\nDamage per block:")
for i, (ds, n) in enumerate(spectrum):
    print(f"  Δσ={ds:5.0f} MPa, n={n:>10,d} → D_i = {va_result.damage_per_block[i]:.6f}")